# Graph Isomorphism and Subgraph Matching

This notebook demonstrates graph isomorphism and subgraph matching techniques
using **topologic_fast**.

## Concept

Graph isomorphism determines if two graphs have the same structure, potentially
with different vertex labels. Subgraph matching finds occurrences of a smaller
"pattern" graph within a larger "host" graph.

Applications include:
- Building comparison and similarity analysis
- Pattern recognition in floor plans
- Finding similar room configurations
- Detecting repeated structural patterns

## Note on topologicpy vs topologic_fast

The original topologicpy has `Graph.Match()` which uses VF2 algorithm for
subgraph isomorphism. In topologic_fast, we demonstrate graph matching concepts
using available graph operations and custom implementations.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

## Helper Functions for Graph Visualization

In [ ]:
def visualize_graph_2d(graph, vertex_labels=None, vertex_colors=None,
                       vertex_sizes=None, edge_color='gray', title='Graph',
                       highlight_vertices=None, highlight_color='red'):
    """
    Visualize a graph in 2D using Plotly.
    
    Parameters:
        graph: tf.Graph
        vertex_labels: dict mapping vertex index to label
        vertex_colors: list of colors for each vertex
        vertex_sizes: list of sizes for each vertex
        highlight_vertices: list of vertex indices to highlight
    """
    fig = go.Figure()
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Create highlight set
    highlight_set = set(highlight_vertices or [])
    
    # Draw edges
    for edge in edges:
        ev = edge.Vertices()
        if len(ev) == 2:
            p1 = ev[0].Coordinates()
            p2 = ev[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color=edge_color, width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        
        # Determine color and size
        if i in highlight_set:
            color = highlight_color
            size = 20
        else:
            color = vertex_colors[i] if vertex_colors else 'blue'
            size = vertex_sizes[i] if vertex_sizes else 12
        
        # Determine label
        label = vertex_labels.get(i, str(i)) if vertex_labels else str(i)
        
        fig.add_trace(go.Scatter(
            x=[coords[0]],
            y=[coords[1]],
            mode='markers+text',
            marker=dict(size=size, color=color, line=dict(color='black', width=1)),
            text=[label],
            textposition='top center',
            textfont=dict(size=10),
            showlegend=False,
            hoverinfo='text',
            hovertext=f'V{i}: ({coords[0]:.1f}, {coords[1]:.1f})'
        ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=600,
        height=500,
        showlegend=False
    )
    
    return fig


def visualize_graph_3d(graph, vertex_colors=None, vertex_size=8, 
                       edge_color='gray', title='Graph 3D',
                       highlight_vertices=None, highlight_color='red'):
    """
    Visualize a graph in 3D using Plotly.
    """
    fig = go.Figure()
    
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    highlight_set = set(highlight_vertices or [])
    
    # Draw edges
    for edge in edges:
        ev = edge.Vertices()
        if len(ev) == 2:
            p1 = ev[0].Coordinates()
            p2 = ev[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color=edge_color, width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices (normal)
    normal_x, normal_y, normal_z = [], [], []
    highlight_x, highlight_y, highlight_z = [], [], []
    
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        if i in highlight_set:
            highlight_x.append(coords[0])
            highlight_y.append(coords[1])
            highlight_z.append(coords[2])
        else:
            normal_x.append(coords[0])
            normal_y.append(coords[1])
            normal_z.append(coords[2])
    
    if normal_x:
        fig.add_trace(go.Scatter3d(
            x=normal_x, y=normal_y, z=normal_z,
            mode='markers',
            marker=dict(size=vertex_size, color='blue'),
            name='Vertices'
        ))
    
    if highlight_x:
        fig.add_trace(go.Scatter3d(
            x=highlight_x, y=highlight_y, z=highlight_z,
            mode='markers',
            marker=dict(size=vertex_size * 1.5, color=highlight_color),
            name='Matched Vertices'
        ))
    
    fig.update_layout(
        title=title,
        scene=dict(aspectmode='data'),
        width=700,
        height=600
    )
    
    return fig

## Graph Signature Functions

For graph comparison, we can compute graph signatures (invariants) that must match
for isomorphic graphs.

In [ ]:
def compute_graph_signature(graph):
    """
    Compute a signature for a graph based on invariants.
    
    Two isomorphic graphs will have the same signature,
    but the converse is not always true (signature matching
    is necessary but not sufficient for isomorphism).
    """
    order = graph.Order()  # Number of vertices
    size = graph.Size()    # Number of edges
    density = graph.Density()
    
    # Degree sequence (sorted)
    degree_seq = sorted(graph.DegreeSequence(), reverse=True)
    
    # Is bipartite
    is_bipartite = graph.IsBipartite()
    
    # Is complete
    is_complete = graph.IsComplete()
    
    # Diameter
    diameter = graph.Diameter()
    
    return {
        'order': order,
        'size': size,
        'density': density,
        'degree_sequence': tuple(degree_seq),
        'is_bipartite': is_bipartite,
        'is_complete': is_complete,
        'diameter': diameter
    }


def signatures_match(sig1, sig2):
    """
    Check if two graph signatures are compatible.
    
    Returns True if the graphs could be isomorphic based on signatures.
    """
    return (
        sig1['order'] == sig2['order'] and
        sig1['size'] == sig2['size'] and
        sig1['degree_sequence'] == sig2['degree_sequence'] and
        sig1['is_bipartite'] == sig2['is_bipartite'] and
        sig1['is_complete'] == sig2['is_complete'] and
        sig1['diameter'] == sig2['diameter']
    )


def print_signature(sig, name='Graph'):
    """Pretty print a graph signature."""
    print(f"{name} Signature:")
    print(f"  Order (vertices): {sig['order']}")
    print(f"  Size (edges): {sig['size']}")
    print(f"  Density: {sig['density']:.3f}")
    print(f"  Degree sequence: {sig['degree_sequence']}")
    print(f"  Is bipartite: {sig['is_bipartite']}")
    print(f"  Is complete: {sig['is_complete']}")
    print(f"  Diameter: {sig['diameter']}")

## Create Test Graphs

Let's create two graphs from prism CellComplexes and check if they are isomorphic.

In [ ]:
# Create a simple CellComplex (2x2 grid)
cells1 = []
for i in range(2):
    for j in range(2):
        cell = tf.Cell.Box(i*2, j*2, 0, 2, 2, 2)
        cells1.append(cell)

cc1 = tf.CellComplex.ByCells(cells1)
g1 = tf.Graph.ByTopology(cc1)

print(f"Graph 1 (from 2x2 grid of cells):")
print(f"  Vertices: {g1.Order()}")
print(f"  Edges: {g1.Size()}")

In [ ]:
# Create a second graph with the same topology but different positions
cells2 = []
for i in range(2):
    for j in range(2):
        # Offset and rotate the grid
        x = 10 + j*2  # Swapped i and j
        y = 10 + i*2
        cell = tf.Cell.Box(x, y, 0, 2, 2, 2)
        cells2.append(cell)

cc2 = tf.CellComplex.ByCells(cells2)
g2 = tf.Graph.ByTopology(cc2)

print(f"Graph 2 (same topology, different position):")
print(f"  Vertices: {g2.Order()}")
print(f"  Edges: {g2.Size()}")

In [ ]:
# Visualize both graphs
fig1 = visualize_graph_3d(g1, title='Graph 1')
fig1.show()

fig2 = visualize_graph_3d(g2, title='Graph 2')
fig2.show()

## Compare Graph Signatures

In [ ]:
# Compute signatures
sig1 = compute_graph_signature(g1)
sig2 = compute_graph_signature(g2)

print_signature(sig1, 'Graph 1')
print()
print_signature(sig2, 'Graph 2')
print()
print(f"Signatures match: {signatures_match(sig1, sig2)}")

## Create a Different Graph

In [ ]:
# Create a 3-cell linear graph (different topology)
cells3 = []
for i in range(3):
    cell = tf.Cell.Box(i*2, 0, 0, 2, 2, 2)
    cells3.append(cell)

cc3 = tf.CellComplex.ByCells(cells3)
g3 = tf.Graph.ByTopology(cc3)

print(f"Graph 3 (linear 3-cell):")
print(f"  Vertices: {g3.Order()}")
print(f"  Edges: {g3.Size()}")

In [ ]:
sig3 = compute_graph_signature(g3)
print_signature(sig3, 'Graph 3')
print()
print(f"Graph 1 vs Graph 3 signatures match: {signatures_match(sig1, sig3)}")

In [ ]:
fig3 = visualize_graph_3d(g3, title='Graph 3 (Linear)')
fig3.show()

## Subgraph Matching

Let's implement a simple subgraph matching algorithm to find occurrences
of a pattern graph within a larger host graph.

**Note:** The topologicpy `Graph.Match()` function uses a sophisticated
VF2-based algorithm. This is a simplified greedy implementation for
demonstration purposes.

In [ ]:
def get_adjacency_structure(graph):
    """
    Get the adjacency structure as a dictionary.
    
    Returns:
        dict mapping vertex index to set of neighbor indices
    """
    vertices = graph.Vertices()
    adj = {}
    
    for i, v in enumerate(vertices):
        neighbors = graph.AdjacentVertices(v)
        neighbor_indices = set()
        
        for neighbor in neighbors:
            n_coords = neighbor.Coordinates()
            # Find index of neighbor
            for j, v2 in enumerate(vertices):
                v2_coords = v2.Coordinates()
                if (abs(n_coords[0] - v2_coords[0]) < 0.01 and
                    abs(n_coords[1] - v2_coords[1]) < 0.01 and
                    abs(n_coords[2] - v2_coords[2]) < 0.01):
                    neighbor_indices.add(j)
                    break
        
        adj[i] = neighbor_indices
    
    return adj, vertices


def find_subgraph_matches(pattern_graph, host_graph, max_matches=10):
    """
    Find occurrences of pattern_graph within host_graph.
    
    This is a simplified backtracking algorithm.
    
    Returns:
        List of mappings (pattern_idx -> host_idx)
    """
    pattern_adj, pattern_verts = get_adjacency_structure(pattern_graph)
    host_adj, host_verts = get_adjacency_structure(host_graph)
    
    n_pattern = len(pattern_verts)
    n_host = len(host_verts)
    
    # Get degree for each vertex
    pattern_degrees = {i: len(neighbors) for i, neighbors in pattern_adj.items()}
    host_degrees = {i: len(neighbors) for i, neighbors in host_adj.items()}
    
    matches = []
    
    def is_compatible(p_idx, h_idx, mapping):
        """Check if pattern vertex p_idx can be mapped to host vertex h_idx."""
        # Check degree compatibility
        if host_degrees[h_idx] < pattern_degrees[p_idx]:
            return False
        
        # Check edge compatibility with already mapped vertices
        for mapped_p, mapped_h in mapping.items():
            # If there's an edge in pattern, there must be an edge in host
            if mapped_p in pattern_adj[p_idx]:
                if mapped_h not in host_adj[h_idx]:
                    return False
        
        return True
    
    def backtrack(mapping):
        """Recursive backtracking."""
        if len(matches) >= max_matches:
            return
        
        if len(mapping) == n_pattern:
            # Found a complete mapping
            matches.append(dict(mapping))
            return
        
        # Get next pattern vertex to map
        p_idx = len(mapping)
        
        # Try mapping to each unmapped host vertex
        mapped_host = set(mapping.values())
        
        for h_idx in range(n_host):
            if h_idx not in mapped_host:
                if is_compatible(p_idx, h_idx, mapping):
                    mapping[p_idx] = h_idx
                    backtrack(mapping)
                    del mapping[p_idx]
    
    backtrack({})
    return matches

## Create Pattern and Host Graphs

In [ ]:
# Create a larger host graph (3x3 grid)
host_cells = []
for i in range(3):
    for j in range(3):
        cell = tf.Cell.Box(i*2, j*2, 0, 2, 2, 2)
        host_cells.append(cell)

host_cc = tf.CellComplex.ByCells(host_cells)
host_graph = tf.Graph.ByTopology(host_cc)

print(f"Host Graph (3x3 grid):")
print(f"  Vertices: {host_graph.Order()}")
print(f"  Edges: {host_graph.Size()}")

In [ ]:
# Create a simple pattern graph (L-shape: 3 cells)
pattern_cells = [
    tf.Cell.Box(0, 0, 0, 2, 2, 2),  # Corner
    tf.Cell.Box(2, 0, 0, 2, 2, 2),  # Right
    tf.Cell.Box(0, 2, 0, 2, 2, 2),  # Up
]

pattern_cc = tf.CellComplex.ByCells(pattern_cells)
pattern_graph = tf.Graph.ByTopology(pattern_cc)

print(f"Pattern Graph (L-shape):")
print(f"  Vertices: {pattern_graph.Order()}")
print(f"  Edges: {pattern_graph.Size()}")

In [ ]:
# Visualize pattern graph
fig = visualize_graph_3d(pattern_graph, title='Pattern Graph (L-shape)')
fig.show()

In [ ]:
# Visualize host graph
fig = visualize_graph_3d(host_graph, title='Host Graph (3x3 grid)')
fig.show()

In [ ]:
# Find matches
matches = find_subgraph_matches(pattern_graph, host_graph, max_matches=20)

print(f"Found {len(matches)} subgraph matches")
for i, match in enumerate(matches[:5]):  # Show first 5
    print(f"  Match {i+1}: {match}")

In [ ]:
# Visualize a match
if matches:
    match = matches[0]
    matched_vertices = set(match.values())
    
    fig = visualize_graph_3d(
        host_graph,
        highlight_vertices=matched_vertices,
        highlight_color='red',
        title=f'Match 1: Host vertices {list(matched_vertices)}'
    )
    fig.show()

## Graph Comparison Using Adjacency Matrices

In [ ]:
def visualize_adjacency_matrix(graph, title='Adjacency Matrix'):
    """
    Visualize the adjacency matrix as a heatmap.
    """
    adj_matrix = graph.AdjacencyMatrix()
    n = len(adj_matrix)
    
    fig = go.Figure(data=go.Heatmap(
        z=adj_matrix,
        x=list(range(n)),
        y=list(range(n)),
        colorscale='Blues',
        showscale=False
    ))
    
    # Add text annotations
    for i in range(n):
        for j in range(n):
            fig.add_annotation(
                x=j, y=i,
                text=str(adj_matrix[i][j]),
                showarrow=False,
                font=dict(color='white' if adj_matrix[i][j] else 'black')
            )
    
    fig.update_layout(
        title=title,
        xaxis=dict(title='Vertex', dtick=1),
        yaxis=dict(title='Vertex', dtick=1, autorange='reversed'),
        width=400,
        height=400
    )
    
    return fig

In [ ]:
# Visualize adjacency matrices
fig1 = visualize_adjacency_matrix(g1, title='Graph 1 Adjacency Matrix')
fig1.show()

fig2 = visualize_adjacency_matrix(g2, title='Graph 2 Adjacency Matrix')
fig2.show()

## Graph Operations: Union, Intersection, Difference

Let's demonstrate graph boolean operations.

In [ ]:
def graph_vertex_set(graph):
    """Get set of vertex coordinates (rounded for comparison)."""
    vertices = graph.Vertices()
    return set(
        (round(v.Coordinates()[0], 3), 
         round(v.Coordinates()[1], 3), 
         round(v.Coordinates()[2], 3))
        for v in vertices
    )


def graph_edge_set(graph):
    """Get set of edge endpoint pairs."""
    edges = graph.Edges()
    edge_set = set()
    
    for edge in edges:
        ev = edge.Vertices()
        if len(ev) == 2:
            c1 = (round(ev[0].Coordinates()[0], 3),
                  round(ev[0].Coordinates()[1], 3),
                  round(ev[0].Coordinates()[2], 3))
            c2 = (round(ev[1].Coordinates()[0], 3),
                  round(ev[1].Coordinates()[1], 3),
                  round(ev[1].Coordinates()[2], 3))
            # Sort to make edge comparison order-independent
            edge_set.add(tuple(sorted([c1, c2])))
    
    return edge_set


# Compare graphs
v_set1 = graph_vertex_set(g1)
v_set2 = graph_vertex_set(g2)

print(f"Graph 1 vertices: {len(v_set1)}")
print(f"Graph 2 vertices: {len(v_set2)}")
print(f"Common vertices: {len(v_set1 & v_set2)}")
print(f"All vertices (union): {len(v_set1 | v_set2)}")

## Semantic Graph Matching

In real applications, vertices often have attributes (room type, area, etc.).
Let's demonstrate matching with vertex labels.

In [ ]:
# Create labeled graphs using tf.Dictionary
# NOTE: Graph vertices in topologic_fast don't directly support dictionaries
# We'll simulate labels with a separate mapping

# Define room types for a 2x2 grid
room_labels_1 = {
    0: 'bedroom',
    1: 'bathroom',
    2: 'kitchen',
    3: 'living'
}

room_labels_2 = {
    0: 'kitchen',      # Swapped
    1: 'living',       # Swapped
    2: 'bedroom',      # Swapped
    3: 'bathroom'      # Swapped
}

print("Room labels for Graph 1:", room_labels_1)
print("Room labels for Graph 2:", room_labels_2)

In [ ]:
def find_labeled_matches(pattern_graph, host_graph, pattern_labels, host_labels, max_matches=10):
    """
    Find subgraph matches respecting vertex labels.
    
    A pattern vertex can only match a host vertex with the same label.
    """
    pattern_adj, pattern_verts = get_adjacency_structure(pattern_graph)
    host_adj, host_verts = get_adjacency_structure(host_graph)
    
    n_pattern = len(pattern_verts)
    n_host = len(host_verts)
    
    pattern_degrees = {i: len(neighbors) for i, neighbors in pattern_adj.items()}
    host_degrees = {i: len(neighbors) for i, neighbors in host_adj.items()}
    
    matches = []
    
    def is_compatible(p_idx, h_idx, mapping):
        # Check label compatibility
        if pattern_labels.get(p_idx) != host_labels.get(h_idx):
            return False
        
        # Check degree compatibility
        if host_degrees[h_idx] < pattern_degrees[p_idx]:
            return False
        
        # Check edge compatibility
        for mapped_p, mapped_h in mapping.items():
            if mapped_p in pattern_adj[p_idx]:
                if mapped_h not in host_adj[h_idx]:
                    return False
        
        return True
    
    def backtrack(mapping):
        if len(matches) >= max_matches:
            return
        
        if len(mapping) == n_pattern:
            matches.append(dict(mapping))
            return
        
        p_idx = len(mapping)
        mapped_host = set(mapping.values())
        
        for h_idx in range(n_host):
            if h_idx not in mapped_host:
                if is_compatible(p_idx, h_idx, mapping):
                    mapping[p_idx] = h_idx
                    backtrack(mapping)
                    del mapping[p_idx]
    
    backtrack({})
    return matches

In [ ]:
# Try matching with same labels (should find a match)
matches_same = find_labeled_matches(g1, g1, room_labels_1, room_labels_1, max_matches=5)
print(f"Same labels - found {len(matches_same)} matches")
for m in matches_same[:3]:
    print(f"  {m}")

In [ ]:
# Try matching with swapped labels
matches_swapped = find_labeled_matches(g1, g2, room_labels_1, room_labels_2, max_matches=5)
print(f"Swapped labels - found {len(matches_swapped)} matches")
for m in matches_swapped:
    print(f"  {m}")

## Summary

This notebook demonstrated:

1. **Graph Creation**: Using `tf.Graph.ByTopology()` to create graphs from CellComplex

2. **Graph Signatures**: Computing invariants for quick compatibility checking

3. **Subgraph Matching**: Finding pattern occurrences in larger graphs

4. **Adjacency Matrices**: Visualizing graph structure

5. **Labeled Matching**: Matching with semantic constraints

### API Differences from topologicpy

| topologicpy | topologic_fast | Notes |
|------------|----------------|-------|
| `Graph.Match()` | Not available | Implement backtracking search as shown |
| `Graph.MeshData()` | Not available | Use `Vertices()` and `Edges()` |
| `Graph.ByMeshData()` | `Graph.ByVerticesEdges()` | Similar API |
| `Graph.Union()` | Not available | Implement using vertex/edge sets |
| `Graph.Intersection()` | Not available | Implement using vertex/edge sets |
| `Graph.Difference()` | Not available | Implement using vertex/edge sets |
| `Dictionary.SetValueAtKey()` | `tf.Dictionary.SetValueAtKey()` | Available |

### Applications

- Building comparison and similarity analysis
- Pattern recognition in floor plans
- Finding similar room configurations
- Detecting repeated structural patterns
- Compliance checking against reference designs
- Architectural style classification